# Week 4 — Final Findings and Business Communication
This notebook verifies the cleaned dataset and five Week 3 chart exports, calculates seven evidence-based findings, proposes three operational investigations, and documents limitations. All comparisons are observational; they do not demonstrate causation. Run from the repository root or the `notebooks/` directory.

## 1. Load data and verify project inputs
Load the same cleaned dataset used throughout Weeks 1–3. Confirm required columns, record counts, duration values, and exported chart availability. The original raw CSV is not required to run this final notebook.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
BASE = Path('.') if Path('data/processed/cleaned_food_delivery.csv').exists() else Path('..')
DATA = BASE/'data/processed/cleaned_food_delivery.csv'
CHARTS = BASE/'visualizations'
assert DATA.is_file(), f'Cleaned dataset missing: {DATA}'
df = pd.read_csv(DATA)
required = ['Time_taken (min)','Road_traffic_density','Weather_conditions','Type_of_vehicle','City','multiple_deliveries','Festival','Restaurant_latitude','Restaurant_longitude','Delivery_location_latitude','Delivery_location_longitude']
assert set(required).issubset(df.columns), f'Missing columns: {set(required)-set(df.columns)}'
df['duration_min'] = pd.to_numeric(df['Time_taken (min)'], errors='coerce')
valid = df.loc[df['duration_min'].notna() & df['duration_min'].ge(0)].copy()
assert len(valid)>0
charts = ['01_delivery_duration_distribution.png','02_traffic_comparison.png','03_weather_comparison.png','04_distance_vs_duration.png','05_vehicle_comparison.png']
checks = pd.DataFrame([{'chart':f, 'exists':(CHARTS/f).is_file(), 'size_bytes':(CHARTS/f).stat().st_size if (CHARTS/f).is_file() else 0} for f in charts])
assert checks['exists'].all() and checks['size_bytes'].gt(0).all(), 'Missing or empty Week 3 charts'
print(f'Cleaned data: {df.shape[0]:,} rows, {df.shape[1]-1} original columns; valid durations: {len(valid):,}')
display(checks)
print('Five PNG files found. Existing notebooks must also be run separately for full reproducibility verification.')

Cleaned data: 45,584 rows, 20 original columns; valid durations: 45,584


,chart,exists,size_bytes
0,01_delivery_duration_distribution.png,True,58782
1,02_traffic_comparison.png,True,65115
2,03_weather_comparison.png,True,95211
3,04_distance_vs_duration.png,True,100119
4,05_vehicle_comparison.png,True,69618


Five PNG files found. Existing notebooks must also be run separately for full reproducibility verification.


## 2. Recalculate the baseline and group comparisons
Recompute the underlying metrics from the cleaned CSV. Group labels with missing values are shown explicitly instead of silently discarded.

In [2]:
duration = valid['duration_min']
baseline = pd.DataFrame({'metric':['count','mean_minutes','median_minutes','minimum_minutes','maximum_minutes','q1_minutes','q3_minutes'], 'value':[duration.count(),duration.mean(),duration.median(),duration.min(),duration.max(),duration.quantile(.25),duration.quantile(.75)]})
display(baseline.round(3))
def grouped(column):
    labels = valid[column].astype('string').str.strip().fillna('Missing').replace('', 'Missing')
    return valid.assign(category=labels).groupby('category')['duration_min'].agg(count='count',mean='mean',median='median').sort_values('mean',ascending=False)
tables = {name:grouped(col) for name,col in [('Traffic','Road_traffic_density'),('Weather','Weather_conditions'),('Vehicle','Type_of_vehicle'),('City','City'),('Multiple deliveries','multiple_deliveries'),('Festival','Festival')]}
for name,table in tables.items():
    display(Markdown('### '+name)); display(table.round(3))

,metric,value
0,count,45584.000
1,mean_minutes,26.294
2,median_minutes,26.000
3,minimum_minutes,10.000
4,maximum_minutes,54.000
5,q1_minutes,19.000
6,q3_minutes,32.000


### Traffic

,count,mean,median
category,,,
Jam,14139,31.176,31.0
High,4423,27.240,27.0
Medium,10945,26.700,27.0
Missing,601,26.542,26.0
Low,15476,21.267,20.0


### Weather

,count,mean,median
category,,,
Cloudy,7533,28.917,28.0
Fog,7653,28.915,28.0
Missing,616,26.545,26.0
Windy,7422,26.119,26.0
Sandstorms,7494,25.876,26.0
Stormy,7584,25.869,26.0
Sunny,7282,21.857,20.0


### Vehicle

... (3 outputs left out)

### Multiple deliveries

,count,mean,median
category,,,
3.0,361,47.820,48.0
2.0,1985,40.455,40.0
1.0,28151,26.855,26.0
0.0,14094,22.876,22.0
Missing,993,22.767,21.0


### Festival

,count,mean,median
category,,,
Yes,896,45.518,45.0
No,44460,25.984,25.0
Missing,228,11.167,11.0


## 3. Verify the distance calculation
Use the same haversine straight-line distance method and coordinate screening as Week 3. Very large calculated distances are flagged, not silently removed; they may reflect coordinate errors. These are not actual route distances.

In [3]:
coords=['Restaurant_latitude','Restaurant_longitude','Delivery_location_latitude','Delivery_location_longitude']
a=valid[coords].apply(pd.to_numeric,errors='coerce')
ok=a.notna().all(axis=1)
for col in ['Restaurant_latitude','Delivery_location_latitude']: ok &= a[col].between(-90,90) & a[col].ne(0)
for col in ['Restaurant_longitude','Delivery_location_longitude']: ok &= a[col].between(-180,180) & a[col].ne(0)
p=a.loc[ok]
lat1=np.radians(p.Restaurant_latitude); lat2=np.radians(p.Delivery_location_latitude)
dlat=lat2-lat1; dlon=np.radians(p.Delivery_location_longitude-p.Restaurant_longitude)
h=np.sin(dlat/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
distance=6371.0*2*np.arcsin(np.sqrt(np.clip(h,0,1)))
pairs=pd.DataFrame({'distance_km':distance,'duration_min':valid.loc[ok,'duration_min']}).dropna()
r=pairs.distance_km.corr(pairs.duration_min)
extreme=int(pairs.distance_km.gt(100).sum())
print(f'Valid coordinate pairs: {len(pairs):,}; Pearson r: {r:.4f}; distances above 100 km: {extreme:,}')
assert np.isfinite(r)

Valid coordinate pairs: 41,944; Pearson r: -0.0029; distances above 100 km: 431


## 4. Seven key findings
Each finding reports an observation, its numerical evidence, a possible operational interpretation, and a limitation. These values are generated directly from the tables above when the notebook runs.

In [4]:
t=tables
jam=t['Traffic'].loc['Jam']; low=t['Traffic'].loc['Low']
cloud=t['Weather'].loc['Cloudy']; sunny=t['Weather'].loc['Sunny']
moto=t['Vehicle'].loc['motorcycle']; es=t['Vehicle'].loc['electric_scooter']
metro=t['City'].loc['Metropolitian']; urban=t['City'].loc['Urban']; semi=t['City'].loc['Semi-Urban']
mult3=t['Multiple deliveries'].loc['3.0']; mult0=t['Multiple deliveries'].loc['0.0']
fest=t['Festival'].loc['Yes']; nofest=t['Festival'].loc['No']
findings=[
 ('Overall delivery-time baseline',f'{len(valid):,} deliveries had mean {duration.mean():.2f} minutes and median {duration.median():.0f} minutes (range {duration.min():.0f}–{duration.max():.0f}).','This provides a descriptive baseline for comparing categories; it is not a contractual service target.'),
 ('Traffic categories differ',f'Jam traffic: median {jam["median"]:.0f} minutes (n={jam["count"]:,}); Low traffic: median {low["median"]:.0f} minutes (n={low["count"]:,}).','Traffic-aware planning is worth investigating, but traffic labels may correlate with location, time, and order mix.'),
 ('Weather categories differ',f'Cloudy weather: mean {cloud["mean"]:.2f} minutes (n={cloud["count"]:,}); Sunny: {sunny["mean"]:.2f} minutes (n={sunny["count"]:,}).','Weather-aware monitoring may help explain variability, but this observational difference is not a causal estimate.'),
 ('Vehicle groups have different averages',f'Motorcycle: mean {moto["mean"]:.2f} minutes (n={moto["count"]:,}); electric scooter: {es["mean"]:.2f} minutes (n={es["count"]:,}).','Vehicle assignment merits review, but routes, delivery loads, and group sizes differ.'),
 ('Multiple-delivery groups show a large gap',f'Three simultaneous deliveries: mean {mult3["mean"]:.2f} minutes (n={mult3["count"]:,}); zero: {mult0["mean"]:.2f} minutes (n={mult0["count"]:,}).','Batching practices warrant investigation; order complexity and assignment selection may confound this difference.'),
 ('Festival orders have longer observed durations',f'Festival=Yes: mean {fest["mean"]:.2f} minutes (n={fest["count"]:,}); Festival=No: {nofest["mean"]:.2f} minutes (n={nofest["count"]:,}).','Peak-day capacity planning may be useful; the festival sample is much smaller and may differ in other respects.'),
 ('Straight-line distance has little overall linear association',f'Pearson r={r:.4f} for {len(pairs):,} valid coordinate pairs; {extreme:,} distances exceed 100 km.','Straight-line coordinates are a weak standalone proxy for delivery duration here; anomalous locations and unmeasured road routes limit interpretation.')]
for i,(title,evidence,interpretation) in enumerate(findings,1):
    display(Markdown(f'### Finding {i}: {title}\n**Observation and numerical evidence:** {evidence}\n\n**Operational interpretation and limitation:** {interpretation}'))

### Finding 1: Overall delivery-time baseline
**Observation and numerical evidence:** 45,584 deliveries had mean 26.29 minutes and median 26 minutes (range 10–54).

**Operational interpretation and limitation:** This provides a descriptive baseline for comparing categories; it is not a contractual service target.

### Finding 2: Traffic categories differ
**Observation and numerical evidence:** Jam traffic: median 31 minutes (n=14,139.0); Low traffic: median 20 minutes (n=15,476.0).

**Operational interpretation and limitation:** Traffic-aware planning is worth investigating, but traffic labels may correlate with location, time, and order mix.

### Finding 3: Weather categories differ
**Observation and numerical evidence:** Cloudy weather: mean 28.92 minutes (n=7,533.0); Sunny: 21.86 minutes (n=7,282.0).

**Operational interpretation and limitation:** Weather-aware monitoring may help explain variability, but this observational difference is not a causal estimate.

### Finding 4: Vehicle groups have different averages
**Observation and numerical evidence:** Motorcycle: mean 27.61 minutes (n=26,429.0); electric scooter: 24.47 minutes (n=3,814.0).

**Operational interpretation and limitation:** Vehicle assignment merits review, but routes, delivery loads, and group sizes differ.

### Finding 5: Multiple-delivery groups show a large gap
**Observation and numerical evidence:** Three simultaneous deliveries: mean 47.82 minutes (n=361.0); zero: 22.88 minutes (n=14,094.0).

**Operational interpretation and limitation:** Batching practices warrant investigation; order complexity and assignment selection may confound this difference.

### Finding 6: Festival orders have longer observed durations
**Observation and numerical evidence:** Festival=Yes: mean 45.52 minutes (n=896.0); Festival=No: 25.98 minutes (n=44,460.0).

**Operational interpretation and limitation:** Peak-day capacity planning may be useful; the festival sample is much smaller and may differ in other respects.

### Finding 7: Straight-line distance has little overall linear association
**Observation and numerical evidence:** Pearson r=-0.0029 for 41,944 valid coordinate pairs; 431 distances exceed 100 km.

**Operational interpretation and limitation:** Straight-line coordinates are a weak standalone proxy for delivery duration here; anomalous locations and unmeasured road routes limit interpretation.

## 5. Three operational recommendations
Recommendations are proposed investigations rather than guaranteed improvements. Validate results using comparable orders and operational context before changing dispatch policies.

In [5]:
recommendations=[
 ('Investigate traffic-aware dispatch and ETA calibration',f'Jam median {jam["median"]:.0f} min (n={jam["count"]:,}) vs Low {low["median"]:.0f} min (n={low["count"]:,}).','Review ETA errors by traffic category and test traffic-informed estimates or dispatch rules.','ETA reliability and dispatch planning.','Compare similar routes, time windows, cities, and order sizes; run a monitored pilot and assess delivery times and customer impact.'),
 ('Review multi-order batching limits',f'Three deliveries mean {mult3["mean"]:.2f} min (n={mult3["count"]:,}) vs zero {mult0["mean"]:.2f} min (n={mult0["count"]:,}).','Audit batch sizes, sequencing, and assignment policies; test adjustments for comparable orders.','Batching and rider workload.','Check whether durations are per-order or per-trip, control for demand and distance, and evaluate pilot effects before rollout.'),
 ('Plan targeted capacity checks for festival periods',f'Festival=Yes mean {fest["mean"]:.2f} min (n={fest["count"]:,}) vs No {nofest["mean"]:.2f} min (n={nofest["count"]:,}).','Investigate staffing and order-load patterns during festival periods and trial targeted capacity planning.','Peak-demand staffing and delivery operations.','Verify festival coding, account for location and demand differences, and compare matched periods before concluding benefit.')]
for i,(title,evidence,action,area,validation) in enumerate(recommendations,1):
 display(Markdown(f'### Recommendation {i}: {title}\n**Observed issue and evidence:** {evidence}\n\n**Proposed action:** {action}\n\n**Potential improvement area:** {area}\n\n**Further validation:** {validation}'))

### Recommendation 1: Investigate traffic-aware dispatch and ETA calibration
**Observed issue and evidence:** Jam median 31 min (n=14,139.0) vs Low 20 min (n=15,476.0).

**Proposed action:** Review ETA errors by traffic category and test traffic-informed estimates or dispatch rules.

**Potential improvement area:** ETA reliability and dispatch planning.

**Further validation:** Compare similar routes, time windows, cities, and order sizes; run a monitored pilot and assess delivery times and customer impact.

### Recommendation 2: Review multi-order batching limits
**Observed issue and evidence:** Three deliveries mean 47.82 min (n=361.0) vs zero 22.88 min (n=14,094.0).

**Proposed action:** Audit batch sizes, sequencing, and assignment policies; test adjustments for comparable orders.

**Potential improvement area:** Batching and rider workload.

**Further validation:** Check whether durations are per-order or per-trip, control for demand and distance, and evaluate pilot effects before rollout.

### Recommendation 3: Plan targeted capacity checks for festival periods
**Observed issue and evidence:** Festival=Yes mean 45.52 min (n=896.0) vs No 25.98 min (n=44,460.0).

**Proposed action:** Investigate staffing and order-load patterns during festival periods and trial targeted capacity planning.

**Potential improvement area:** Peak-demand staffing and delivery operations.

**Further validation:** Verify festival coding, account for location and demand differences, and compare matched periods before concluding benefit.

## 6. Conclusion and limitations
The cleaned dataset supports descriptive comparisons of delivery duration across traffic, weather, vehicle, batching, and festival categories. These comparisons are useful for prioritizing operational investigations, not for claiming that changing one category will necessarily change delivery times.

Limitations: the analysis is observational; category sizes vary; some fields are missing; partner years of experience is not provided; straight-line distances are not road-route distances; extreme coordinate-derived distances may be data errors; and undocumented category codes may require confirmation. The original raw data is not redistributed without permission. Next steps include validating definitions and data quality, comparing matched orders, and conducting monitored operational pilots.

**Reproducibility note:** This notebook checks the five existing PNG exports and recalculates key values from the cleaned dataset. To satisfy the full Kagnus Week 4 verification requirement, run each of the earlier notebooks separately from first cell to last in the documented environment and compare regenerated charts with the committed PNGs.